# Part 5 — Attack Artifact Evaluation (Table 2)

Run **PAIR, GCG, JBC (AIM), prompt_with_random_search** attack artifacts against
Llama-2-7B-chat (FP16) and report ASR via **Llama-Guard-3-8B** (same judge as
Parts 6, 8, 9 — gives a uniform ASR taxonomy across the entire deliverable).

Two vLLM loads in this kernel: target (Llama-2-7B-chat) → Guard-3-8B. Each is
freed before the next is constructed.

**Outputs**: `results_part5/summary.csv`, `results_part5/raw.json`.


In [ ]:
import os, sys
from pathlib import Path

# Self-locate jbb_common.py so kernel CWD doesn't matter (tml/ vs repo root).
_cwd = Path.cwd()
if (_cwd / "jbb_common.py").exists():
    pass
elif (_cwd / "tml" / "jbb_common.py").exists():
    os.chdir(_cwd / "tml")
else:
    raise FileNotFoundError(f"jbb_common.py not found from {_cwd}; run from tml/ or repo root.")
sys.path.insert(0, str(Path.cwd()))

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


In [ ]:
# ---- Phase 1: target generation (Llama-2-7B-chat) ----
import os, json, gc
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import jailbreakbench as jbb

os.makedirs("results_part5", exist_ok=True)

METHODS = ["PAIR", "GCG", "JBC", "prompt_with_random_search"]

llm_attack = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    tensor_parallel_size=1,
    enforce_eager=True,
)
attack_sp = SamplingParams(temperature=0.0, max_tokens=150)

gen_per_method = {}
for method in METHODS:
    try:
        artifact = jbb.read_artifact(method=method, model_name="llama-2-7b-chat-hf")
    except ValueError as e:
        print(f"Skipping {method}: artifact not found ({e})")
        continue

    valid   = [jb for jb in artifact.jailbreaks if jb is not None and jb.prompt is not None]
    rows    = [{"behavior": jb.behavior, "goal": jb.goal, "prompt": jb.prompt}
               for jb in valid]
    if not rows:
        print(f"{method:30s} no valid prompts")
        continue

    prompts   = [r["prompt"] for r in rows]
    outs      = llm_attack.generate(prompts, attack_sp)
    responses = [o.outputs[0].text for o in outs]
    gen_per_method[method] = [
        {**r, "response": resp} for r, resp in zip(rows, responses)
    ]
    print(f"gen / {method:30s}  n={len(rows)}")

del llm_attack
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")


In [ ]:
# ---- Phase 2: Llama-Guard-3-8B judging ----
JUDGE_MODEL = "meta-llama/Llama-Guard-3-8B"
guard3_llm = LLM(
    model=JUDGE_MODEL, dtype="float16",
    gpu_memory_utilization=0.90, max_model_len=2048, enforce_eager=True,
)
guard3_tok    = AutoTokenizer.from_pretrained(JUDGE_MODEL)
guard3_params = SamplingParams(temperature=0.0, max_tokens=20)

def judge_batch(prompt_response_pairs):
    token_id_lists = [
        guard3_tok.apply_chat_template(
            [{"role": "user", "content": p}, {"role": "assistant", "content": r}],
            tokenize=True,
        )
        for (p, r) in prompt_response_pairs
    ]
    outs = guard3_llm.generate(
        [{"prompt_token_ids": ids} for ids in token_id_lists],
        guard3_params,
    )
    return [1 if o.outputs[0].text.strip().lower().startswith("unsafe") else 0
            for o in outs]

attack_results = []
raw_payload    = {}
for method, rows in gen_per_method.items():
    pairs  = [(r["goal"], r["response"]) for r in rows]
    labels = judge_batch(pairs)
    asr    = sum(labels) / max(len(labels), 1)
    attack_results.append({
        "Method":               method,
        "ASR":                  round(asr, 4),
        "Total Prompts Tested": len(rows),
    })
    raw_payload[method] = [
        {"prompt": r["prompt"], "goal": r["goal"], "response": r["response"],
         "judged_unsafe": bool(lab)}
        for r, lab in zip(rows, labels)
    ]
    print(f"{method:30s}  ASR={asr:.1%}  Prompts={len(rows)}")

del guard3_llm
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

summary_df = pd.DataFrame(attack_results)
summary_df.to_csv("results_part5/summary.csv", index=False)
with open("results_part5/raw.json", "w") as f:
    json.dump(raw_payload, f, indent=2)

print("\n### Table 2: Attack Evaluation on Llama-2-7B-chat (FP16, Llama-Guard-3-8B judge) ###")
print(summary_df.to_string(index=False))
